# 10. XGBoost 하이퍼파라미터 탐색

## 배경
- 09에서 XGBoost(제공 베이스라인 설정)로 LB 0.2995 (LR 0.2739 대비 _+0.0256)
- 기본 설정은 깊이 6, 규제 없음 → 피처 4,393개·6,201행에서 과적합 가능성

## 가설
- H10: 나무를 얕게 하고 규제를 걸면 과적합이 줄어 CV와 LB가 오른다

## 검증 방법
- 1단계: 깊이 × 학습률 (나무 수는 학습률에 맞춰 조정)
- 2단계: 1단계 최고 조합에 열 샘플링·L2 규제 추가
- 판단 기준: CV 평균뿐 아니라 std도 함께 봄 (07에서 CV만 보고 고르면 LB와 어긋남을 확인)
- LB 제출은 최종 후보 1개만

#### 준비: 데이터, 래퍼, 공통 평가 함수
- 피처와 `BalancedXGB`는 09와 동일 (피처 고정, 하이퍼파라미터만 비교)
- `BASE`: 탐색하지 않는 고정 옵션 (트리 방식, 시드 등)
- `evaluate()`: 설정 하나를 받아 공통 CV를 돌리고, 이름·설정값·CV·std·소요 시간을 한 줄로 기록
  - 모든 설정을 같은 함수로 평가해서 결과 표를 바로 비교할 수 있게 함

In [1]:
import sys; sys.path.append('..')
import time, pandas as pd, xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)
X = pd.concat([F_tr['any'], F_tr['stats']], axis=1)
X_test = pd.concat([F_te['any'], F_te['stats']], axis=1)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BASE = dict(tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

def evaluate(name, params):
    t = time.time()
    r = run_cv(lambda: BalancedXGB(**BASE, **params), X, y, verbose=False)
    row = dict(name=name, **params, cv=round(r['cv_mean'], 4), std=round(r['cv_std'], 4),
               sec=round(time.time() - t))
    print(row)
    return row

#### 1단계: 나무 깊이 × 학습률
- 가장 영향이 큰 두 값부터 탐색
  - `max_depth`: 깊을수록 복잡한 조합을 배우지만 과적합 위험 (피처 4,393개, 환자 6,201명)
  - `learning_rate`: 작을수록 천천히 안정적으로 학습 → 대신 나무 수를 늘려야 함
- 학습률을 0.1 → 0.05로 줄인 조합은 나무 수를 2~3배로 맞춤 (학습량을 비슷하게 유지)
- `d6_lr0.1`은 09와 같은 설정 → 기준선

In [2]:
# 1단계: 깊이 × 학습률 (학습률을 반으로 줄이면 나무 수를 2~3배로)
step1 = [
    ('d6_lr0.1',  dict(max_depth=6, learning_rate=0.1,  n_estimators=100)),  # 09 기준
    ('d4_lr0.1',  dict(max_depth=4, learning_rate=0.1,  n_estimators=150)),
    ('d3_lr0.1',  dict(max_depth=3, learning_rate=0.1,  n_estimators=200)),
    ('d6_lr0.05', dict(max_depth=6, learning_rate=0.05, n_estimators=250)),
    ('d4_lr0.05', dict(max_depth=4, learning_rate=0.05, n_estimators=300)),
]
res1 = pd.DataFrame([evaluate(n, p) for n, p in step1]).sort_values('cv', ascending=False)
res1

{'name': 'd6_lr0.1', 'max_depth': 6, 'learning_rate': 0.1, 'n_estimators': 100, 'cv': 0.3879, 'std': 0.0058, 'sec': 109}
{'name': 'd4_lr0.1', 'max_depth': 4, 'learning_rate': 0.1, 'n_estimators': 150, 'cv': 0.4021, 'std': 0.0072, 'sec': 123}
{'name': 'd3_lr0.1', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'cv': 0.4054, 'std': 0.0127, 'sec': 138}
{'name': 'd6_lr0.05', 'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 250, 'cv': 0.3914, 'std': 0.0079, 'sec': 222}
{'name': 'd4_lr0.05', 'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 300, 'cv': 0.4004, 'std': 0.0076, 'sec': 228}


,name,max_depth,learning_rate,n_estimators,cv,std,sec
2,d3_lr0.1,3,0.10,200,0.4054,0.0127,138
1,d4_lr0.1,4,0.10,150,0.4021,0.0072,123
4,d4_lr0.05,4,0.05,300,0.4004,0.0076,228
3,d6_lr0.05,6,0.05,250,0.3914,0.0079,222
0,d6_lr0.1,6,0.10,100,0.3879,0.0058,109


#### 2단계: 규제 파라미터를 하나씩, 여러 값으로 비교
- 1단계 최고 조합을 고정하고, 규제 파라미터를 **한 번에 하나씩** 바꿔서 각각의 효과를 따로 확인
  - `colsample_bytree` (0.3 / 0.5 / 0.7): 나무마다 피처를 일부만 보게 함 → 특정 피처 의존을 줄임
  - `reg_lambda` (1 / 3 / 10): 잎 값이 너무 커지지 않게 누름 (L2 규제)
  - `min_child_weight` (1 / 3 / 10): 잎 하나에 최소한 들어가야 하는 데이터 양
- 탐색 범위는 각 파라미터의 일반적인 범위에서 고르게 선택
- `lambda1.0`, `min_child1`은 XGBoost 기본값 → `best`와 같은 점수가 나오는지로 코드 정상 여부 확인

In [3]:
# 2단계: 1단계 최고 조합에서 규제 파라미터를 하나씩, 여러 값으로 비교
best = step1[[n for n, _ in step1].index(res1.iloc[0]['name'])][1]
step2 = [('best', best)]
for v in [0.3, 0.5, 0.7]:
    step2.append((f'colsample{v}', {**best, 'colsample_bytree': v}))
for v in [1.0, 3.0, 10.0]:
    step2.append((f'lambda{v}', {**best, 'reg_lambda': v}))
for v in [1, 3, 10]:
    step2.append((f'min_child{v}', {**best, 'min_child_weight': v}))

res2 = pd.DataFrame([evaluate(n, p) for n, p in step2]).sort_values('cv', ascending=False)
res2

{'name': 'best', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'cv': 0.4054, 'std': 0.0127, 'sec': 146}
{'name': 'colsample0.3', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'colsample_bytree': 0.3, 'cv': 0.4113, 'std': 0.0024, 'sec': 144}
{'name': 'colsample0.5', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'colsample_bytree': 0.5, 'cv': 0.4079, 'std': 0.0094, 'sec': 150}
{'name': 'colsample0.7', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'colsample_bytree': 0.7, 'cv': 0.4093, 'std': 0.0077, 'sec': 151}
{'name': 'lambda1.0', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'reg_lambda': 1.0, 'cv': 0.4054, 'std': 0.0127, 'sec': 138}
{'name': 'lambda3.0', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'reg_lambda': 3.0, 'cv': 0.4089, 'std': 0.0064, 'sec': 149}
{'name': 'lambda10.0', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'reg_lambda': 10.0, 'cv': 0.3959, 'std': 0.0043, 'sec': 151}
{'name':

,name,max_depth,learning_rate,n_estimators,cv,std,sec,colsample_bytree,reg_lambda,min_child_weight
1,colsample0.3,3,0.1,200,0.4113,0.0024,144,0.3,NaN,NaN
3,colsample0.7,3,0.1,200,0.4093,0.0077,151,0.7,NaN,NaN
5,lambda3.0,3,0.1,200,0.4089,0.0064,149,NaN,3.0,NaN
2,colsample0.5,3,0.1,200,0.4079,0.0094,150,0.5,NaN,NaN
0,best,3,0.1,200,0.4054,0.0127,146,NaN,NaN,NaN
4,lambda1.0,3,0.1,200,0.4054,0.0127,138,NaN,1.0,NaN
7,min_child1,3,0.1,200,0.4054,0.0127,137,NaN,NaN,1.0
8,min_child3,3,0.1,200,0.4035,0.0047,139,NaN,NaN,3.0
6,lambda10.0,3,0.1,200,0.3959,0.0043,151,NaN,10.0,NaN
9,min_child10,3,0.1,200,0.3766,0.0097,150,NaN,NaN,10.0


#### 3단계: 효과 있던 값 합치기
- 2단계에서 단독으로 효과 있던 colsample 0.3, lambda 3을 합침
- `colsample0.3` 단독도 같이 비교 → 합쳤을 때 오히려 떨어지거나 불안정해지면 단독을 선택

In [4]:
# 3단계: 2단계에서 각각 효과 있던 값만 합쳐서 확인
# 합쳤을 때 오히려 점수가 떨어진다면 colsample_bytree 0.3 단독 선택
combo = {**best, 'colsample_bytree': 0.3, 'reg_lambda': 3.0}
res3 = pd.DataFrame([evaluate('best', best),
                     evaluate('colsample0.3', {**best, 'colsample_bytree': 0.3}),
                     evaluate('combo', combo)])
res3

{'name': 'best', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'cv': 0.4054, 'std': 0.0127, 'sec': 146}
{'name': 'colsample0.3', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'colsample_bytree': 0.3, 'cv': 0.4113, 'std': 0.0024, 'sec': 139}
{'name': 'combo', 'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200, 'colsample_bytree': 0.3, 'reg_lambda': 3.0, 'cv': 0.4136, 'std': 0.009, 'sec': 136}


,name,max_depth,learning_rate,n_estimators,cv,std,sec,colsample_bytree,reg_lambda
0,best,3,0.1,200,0.4054,0.0127,146,NaN,NaN
1,colsample0.3,3,0.1,200,0.4113,0.0024,139,0.3,NaN
2,combo,3,0.1,200,0.4136,0.0090,136,0.3,3.0


#### 최종 후보 제출
- 3단계 결과로 고른 설정 1개만 제출 (하루 제출 4회 제한)
- 선택 기준: CV 차이가 std보다 작으면 사실상 같은 점수 → std가 작은 쪽(test에서 덜 흔들릴 쪽)을 고름

In [5]:
# 최종 후보로 제출 파일 생성
CHOSEN = 'colsample0.3'
params = dict(step2)[CHOSEN]
r = run_cv(lambda: BalancedXGB(**BASE, **params), X, y, X_test)
save_submission(submission, r['test_proba'], le, name=f'xgb_d3_{CHOSEN}', cv_score=r['cv_mean'])

[fold 0] Macro F1 = 0.4088
[fold 1] Macro F1 = 0.4156
[fold 2] Macro F1 = 0.4117
[fold 3] Macro F1 = 0.4094
[fold 4] Macro F1 = 0.4109
CV Macro F1 = 0.4113 ± 0.0024 (OOF 0.4120)
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1110_xgb_d3_colsample0.3_0.411.csv


WindowsPath('C:/Users/USER/Desktop/hackathon/submissions/1110_xgb_d3_colsample0.3_0.411.csv')

## 결과

**1단계** (깊이 × 학습률): 최고 `d3_lr0.1` (depth 3, lr 0.1, 200그루) **0.4054 ± 0.0127** → 09(0.3879) 대비 +0.018
- 얕은 나무가 가장 좋음 → 기본 깊이 6은 과적합이었던 것으로 보임

**2단계** (1단계 최고 기준, 한 번에 하나씩)

| 설정 | CV (± std) | best 대비 | 판단 |
| :--- | :-: | :-: | :--- |
| best (기준) | 0.4054 ± 0.0127 | — | |
| **colsample 0.3** | **0.4113 ± 0.0024** | **+0.006** | std 최소 |
| colsample 0.7 | 0.4093 ± 0.0077 | +0.004 | |
| colsample 0.5 | 0.4079 ± 0.0094 | +0.003 | |
| lambda 3 | 0.4089 ± 0.0064 | +0.004 | |
| lambda 10 | 0.3959 ± 0.0043 | −0.010 | 규제 과다 |
| min_child 3 | 0.4035 ± 0.0047 | −0.002 | 변화 없음 |
| min_child 10 | 0.3766 ± 0.0097 | −0.029 | 기각 |

- lambda 1, min_child 1(기본값)이 best와 같은 점수 → 코드 정상 확인

**3단계**: colsample 0.3 + lambda 3 = 0.4136 ± 0.0090 (단독 대비 +0.002, std 4배)

**제출**: colsample 0.3 단독 → **LB 0.3206** (09 대비 +0.021)

## 결론
- **H10 채택**: 얕은 나무 + 피처 샘플링으로 CV +0.023, LB +0.021
- **colsample이 가장 효과적**: 나무마다 유전자를 30%만 보게 하니 특정 피처에 덜 의존 → 점수와 안정성 모두 개선
- **min_child_weight는 올릴수록 손해**: 소수 아형(DLBC 38명 등)은 잎에 들어가는 환자가 적어서, 값을 키우면 그 아형을 학습하지 못하는 것으로 추정
- 합친 조합은 점수 차이가 오차 범위이고 std가 커져서, 단순하고 안정적인 colsample 0.3 단독을 선택
- CV−LB 차이: 0.088 → 0.091로 유지 (튜닝이 train에만 과적합되지 않았음)

## 다음
- 11: 최적 설정 고정 후 피처 추가 (핫스팟 변이)